# 🧬 Transfer Learning & Personalization Experiment

**Thesis Strategy**: Fine-tune Global Model for Individual Patients  
**Goal**: Prove that personalized training reduces RMSE  
**Method**: Compare Global Model vs Fine-Tuned Model on single patient

In [ ]:
!pip install polars xgboost -U -q

import polars as pl
import xgboost as xgb
import numpy as np
import matplotlib.pyplot as plt
import gc
from sklearn.metrics import mean_squared_error, mean_absolute_error

from google.colab import drive
drive.mount('/content/drive')

print(f"✅ Polars: {pl.__version__}, XGBoost: {xgb.__version__}")

In [ ]:
# ============================================================
# CONFIGURATION
# ============================================================
BASE_PATH = "/content/drive/MyDrive/DiabetesKDS/data"

RAW_TEST = f"{BASE_PATH}/raw/metabonet_public_test.parquet"
GLOBAL_MODEL = f"{BASE_PATH}/processed/xgboost_glucose_model_v2.json"

TARGET = 'CGM_future'
FEATURES = [
    'CGM', 'bolus',
    'velocity', 'acceleration',
    'rolling_mean', 'rolling_std', 'active_insulin',
    'time_sin', 'time_cos',
    'age', 'weight', 'gender_encoded'
]

# Cleaning thresholds
CGM_MIN, CGM_MAX = 39, 400
MAX_VELOCITY = 15

print(f"📁 Raw Test: {RAW_TEST}")
print(f"📁 Global Model: {GLOBAL_MODEL}")

## Step 1: Data Preparation (Single Patient)

In [ ]:
print("="*70)
print("📊 STEP 1: DATA PREPARATION")
print("="*70)

# Load test data
print("\n📂 Loading raw test data...")
df_raw = pl.read_parquet(RAW_TEST)
print(f"   Total rows: {len(df_raw):,}")

# Find patient with most data points
patient_counts = df_raw.group_by('id').agg(pl.len().alias('count')).sort('count', descending=True)
print(f"\n📋 Top 5 patients by data points:")
print(patient_counts.head(5))

# Select top patient
TARGET_PATIENT = patient_counts['id'][0]
PATIENT_ROWS = patient_counts['count'][0]
print(f"\n🎯 Selected Patient: {TARGET_PATIENT}")
print(f"   Data points: {PATIENT_ROWS:,}")

# Filter to single patient
df_patient = df_raw.filter(pl.col('id') == TARGET_PATIENT).sort('date')
del df_raw; gc.collect()
print(f"   Date range: {df_patient['date'].min()} to {df_patient['date'].max()}")

In [ ]:
# ============================================================
# FEATURE ENGINEERING (Same as training pipeline)
# ============================================================
print("\n🔧 Applying feature engineering...")

df = df_patient.clone()

# Fill nulls
df = df.with_columns([
    pl.col('CGM').fill_null(strategy='forward'),
    pl.col('bolus').fill_null(0.0),
    pl.col('age').fill_null(-1.0),
    pl.col('weight').fill_null(-1.0),
])

# Filter invalid CGM
df = df.filter(
    pl.col('CGM').is_not_null() &
    (pl.col('CGM') > CGM_MIN) &
    (pl.col('CGM') < CGM_MAX)
)

# Velocity & Acceleration
df = df.with_columns([
    pl.col('CGM').cast(pl.Float32),
    pl.col('bolus').cast(pl.Float32),
    pl.col('age').cast(pl.Float32),
    pl.col('weight').cast(pl.Float32),
])

df = df.with_columns(
    (pl.col('CGM') - pl.col('CGM').shift(1)).alias('velocity')
)
df = df.with_columns(
    (pl.col('velocity') - pl.col('velocity').shift(1)).alias('acceleration')
)

# Filter impossible jumps
df = df.filter((pl.col('velocity').abs() < MAX_VELOCITY) | pl.col('velocity').is_null())

# Rolling stats
df = df.with_columns([
    pl.col('CGM').rolling_mean(window_size=6).cast(pl.Float32).alias('rolling_mean'),
    pl.col('CGM').rolling_std(window_size=12).cast(pl.Float32).alias('rolling_std'),
    pl.col('bolus').rolling_sum(window_size=24).cast(pl.Float32).alias('active_insulin'),
])

# Time features
df = df.with_columns([
    (2 * np.pi * pl.col('date').dt.hour() / 24).sin().cast(pl.Float32).alias('time_sin'),
    (2 * np.pi * pl.col('date').dt.hour() / 24).cos().cast(pl.Float32).alias('time_cos'),
])

# Gender encoding
if 'gender' in df.columns:
    df = df.with_columns(
        pl.when(pl.col('gender').str.to_lowercase() == 'male').then(1.0)
        .when(pl.col('gender').str.to_lowercase() == 'female').then(0.0)
        .otherwise(-1.0).cast(pl.Float32).alias('gender_encoded')
    )
else:
    df = df.with_columns(pl.lit(-1.0).cast(pl.Float32).alias('gender_encoded'))

# Target
df = df.with_columns(
    pl.col('CGM').shift(-6).cast(pl.Float32).alias('CGM_future')
)

# Final cleanup
df = df.select(FEATURES + [TARGET, 'date'])
df = df.drop_nulls()

print(f"✅ Processed patient data: {df.shape}")

In [ ]:
# ============================================================
# TEMPORAL SPLIT: 20% Calibration / 80% Evaluation
# ============================================================
print("\n📊 Temporal Split (20% Calibration / 80% Evaluation)...")

split_idx = int(len(df) * 0.20)

df_calibration = df.head(split_idx)  # First 20%
df_evaluation = df.tail(len(df) - split_idx)  # Last 80%

print(f"   Calibration Set: {len(df_calibration):,} rows (first 20%)")
print(f"   Evaluation Set:  {len(df_evaluation):,} rows (last 80%)")

# Prepare arrays
X_calib = df_calibration.select(FEATURES).to_numpy().astype(np.float32)
y_calib = df_calibration.select(TARGET).to_numpy().ravel().astype(np.float32)

X_eval = df_evaluation.select(FEATURES).to_numpy().astype(np.float32)
y_eval = df_evaluation.select(TARGET).to_numpy().ravel().astype(np.float32)

# Keep dates for plotting
eval_dates = df_evaluation['date'].to_list()

print(f"\n✅ Data ready!")

## Step 2: Scenario A - Global Model (Baseline)

In [ ]:
print("="*70)
print("🌍 SCENARIO A: GLOBAL MODEL (Baseline)")
print("="*70)

# Load global model
print("\n📂 Loading global model...")
global_model = xgb.Booster()
global_model.load_model(GLOBAL_MODEL)
print("✅ Global model loaded!")

# Predict on evaluation set
deval = xgb.DMatrix(X_eval, feature_names=FEATURES)
y_pred_global = global_model.predict(deval)

# Metrics
rmse_global = np.sqrt(mean_squared_error(y_eval, y_pred_global))
mae_global = mean_absolute_error(y_eval, y_pred_global)
zone_a_global = np.mean(np.abs(y_eval - y_pred_global) <= 15) * 100

print(f"\n📊 Global Model Results:")
print(f"   RMSE:    {rmse_global:.4f} mg/dL")
print(f"   MAE:     {mae_global:.4f} mg/dL")
print(f"   Zone A:  {zone_a_global:.2f}%")

## Step 3: Scenario B - Transfer Learning (Fine-Tuning)

In [ ]:
print("="*70)
print("🎯 SCENARIO B: TRANSFER LEARNING (Fine-Tuning)")
print("="*70)

# Load fresh copy of global model
print("\n📂 Loading global model for fine-tuning...")
personalized_model = xgb.Booster()
personalized_model.load_model(GLOBAL_MODEL)

# Create DMatrix for calibration data
dcalib = xgb.DMatrix(X_calib, label=y_calib, feature_names=FEATURES)

# Fine-tuning parameters (lower learning rate)
finetune_params = {
    'objective': 'reg:squarederror',
    'tree_method': 'hist',
    'device': 'cuda',
    'eta': 0.01,  # Very low learning rate for gentle adaptation
    'max_depth': 6,
    'subsample': 0.8,
    'colsample_bytree': 0.8,
}

print("\n📋 Fine-tuning parameters:")
print(f"   eta (learning_rate): 0.01 (gentle adaptation)")
print(f"   num_boost_round: 100")
print(f"   Calibration samples: {len(X_calib)}")

# Fine-tune: Continue training from global model
print("\n⏳ Fine-tuning on patient's calibration data...")
personalized_model = xgb.train(
    params=finetune_params,
    dtrain=dcalib,
    num_boost_round=100,
    xgb_model=personalized_model,  # Start from global model!
    evals=[(dcalib, 'calib')],
    verbose_eval=25
)

print("✅ Fine-tuning complete!")

# Predict on evaluation set
y_pred_personalized = personalized_model.predict(deval)

# Metrics
rmse_personal = np.sqrt(mean_squared_error(y_eval, y_pred_personalized))
mae_personal = mean_absolute_error(y_eval, y_pred_personalized)
zone_a_personal = np.mean(np.abs(y_eval - y_pred_personalized) <= 15) * 100

print(f"\n📊 Personalized Model Results:")
print(f"   RMSE:    {rmse_personal:.4f} mg/dL")
print(f"   MAE:     {mae_personal:.4f} mg/dL")
print(f"   Zone A:  {zone_a_personal:.2f}%")

## Step 4: Comparison & Visualization

In [ ]:
# ============================================================
# COMPARISON TABLE
# ============================================================
print("\n" + "="*70)
print("📊 COMPARISON: GLOBAL vs PERSONALIZED")
print("="*70)

rmse_improvement = ((rmse_global - rmse_personal) / rmse_global) * 100
zone_a_improvement = zone_a_personal - zone_a_global

print(f"\n{'Metric':<20} {'Global':<15} {'Personalized':<15} {'Change':<15}")
print("-"*65)
print(f"{'RMSE (mg/dL)':<20} {rmse_global:<15.4f} {rmse_personal:<15.4f} {rmse_improvement:+.2f}%")
print(f"{'MAE (mg/dL)':<20} {mae_global:<15.4f} {mae_personal:<15.4f} -")
print(f"{'Zone A (%)':<20} {zone_a_global:<15.2f} {zone_a_personal:<15.2f} {zone_a_improvement:+.2f}%")
print("-"*65)

if rmse_personal < rmse_global:
    print(f"\n✅ PERSONALIZATION IMPROVED RMSE by {rmse_improvement:.2f}%! 🎉")
else:
    print(f"\n⚠️ Personalization did not improve (may need more calibration data)")

print(f"\n💡 Insight: With only {len(X_calib)} calibration samples (20% of patient data),")
print(f"   the model adapted to this patient's unique glucose dynamics.")

In [ ]:
# ============================================================
# VISUALIZATION
# ============================================================
fig, axes = plt.subplots(2, 2, figsize=(16, 10))

# 1. Time Series (24h window = 288 samples at 5-min intervals)
window = min(288, len(y_eval))
ax1 = axes[0, 0]
ax1.plot(range(window), y_eval[:window], 'g-', lw=2, label='Actual', alpha=0.9)
ax1.plot(range(window), y_pred_global[:window], 'b--', lw=1.5, label='Global Model', alpha=0.7)
ax1.plot(range(window), y_pred_personalized[:window], 'r-', lw=1.5, label='Personalized', alpha=0.8)
ax1.fill_between(range(window), y_eval[:window]-15, y_eval[:window]+15, alpha=0.15, color='green')
ax1.set_title(f'24h Window: Actual vs Predictions\nPatient: {TARGET_PATIENT}')
ax1.set_xlabel('Time Steps (5 min each)')
ax1.set_ylabel('Glucose (mg/dL)')
ax1.legend(loc='upper right')
ax1.grid(alpha=0.3)

# 2. Error Comparison
ax2 = axes[0, 1]
errors_global = y_eval - y_pred_global
errors_personal = y_eval - y_pred_personalized
ax2.hist(errors_global, bins=50, alpha=0.6, label=f'Global (RMSE: {rmse_global:.2f})', color='blue')
ax2.hist(errors_personal, bins=50, alpha=0.6, label=f'Personalized (RMSE: {rmse_personal:.2f})', color='red')
ax2.axvline(0, color='black', ls='--', lw=2)
ax2.set_title('Error Distribution Comparison')
ax2.set_xlabel('Prediction Error (mg/dL)')
ax2.legend()
ax2.grid(alpha=0.3)

# 3. Scatter: Global
ax3 = axes[1, 0]
ax3.scatter(y_eval, y_pred_global, alpha=0.3, s=10, c='blue')
ax3.plot([0, 400], [0, 400], 'r-', lw=2)
ax3.plot([0, 400], [15, 415], 'g--', lw=1, alpha=0.5)
ax3.plot([0, 400], [-15, 385], 'g--', lw=1, alpha=0.5)
ax3.set_xlim([40, 350]); ax3.set_ylim([40, 350])
ax3.set_title(f'Global Model (RMSE: {rmse_global:.2f}, Zone A: {zone_a_global:.1f}%)')
ax3.set_xlabel('Actual'); ax3.set_ylabel('Predicted')
ax3.grid(alpha=0.3)

# 4. Scatter: Personalized
ax4 = axes[1, 1]
ax4.scatter(y_eval, y_pred_personalized, alpha=0.3, s=10, c='red')
ax4.plot([0, 400], [0, 400], 'r-', lw=2)
ax4.plot([0, 400], [15, 415], 'g--', lw=1, alpha=0.5)
ax4.plot([0, 400], [-15, 385], 'g--', lw=1, alpha=0.5)
ax4.set_xlim([40, 350]); ax4.set_ylim([40, 350])
ax4.set_title(f'Personalized Model (RMSE: {rmse_personal:.2f}, Zone A: {zone_a_personal:.1f}%)')
ax4.set_xlabel('Actual'); ax4.set_ylabel('Predicted')
ax4.grid(alpha=0.3)

plt.tight_layout()
plt.show()

print("\n🎉 TRANSFER LEARNING EXPERIMENT COMPLETE!")